In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv

In [3]:
# --- 1. CONFIGURATION ---

# Path to your trained model file
MODEL_PATH = "car_detection_model.pt"

# Path to the image you want to test
IMAGE_PATH = "sample_vehicle_image.jpg" 

# --- FILTER THRESHOLDS ---
# 1. Minimum confidence score for a detection to be considered valid (e.g., 75%)
CONFIDENCE_THRESHOLD = 0.75 

# 2. Minimum bounding box area as a ratio of the total image area.
# This filters out small, distant background cars. (e.g., must occupy at least 10% of the image)
MIN_NORMALIZED_AREA_RATIO = 0.10 

In [ ]:

# --- 2. LOAD MODEL AND IMAGE ---

# Load the custom trained YOLO model
model = YOLO(MODEL_PATH)

# Load the image using OpenCV
image = cv2.imread(IMAGE_PATH)


# Get image dimensions
H, W, _ = image.shape
TOTAL_IMAGE_AREA = H * W

print(f"Image Dimensions: {W}x{H} (Total Area: {TOTAL_IMAGE_AREA} pixels)")

Image Dimensions: 612x408 (Total Area: 249696 pixels)


In [10]:
# --- 3. RUN INFERENCE ---

# Run model prediction
# The 'conf' argument here sets the *initial* model output threshold,
# but we will apply a stricter one later via supervision.
results = model(image, conf=0.25, verbose=False)[0] 

# --- 4. POST-PROCESSING (Filtering using Supervision) ---

# Convert YOLO results into a Supervision Detections object
detections = sv.Detections.from_ultralytics(results)

# --- 4a. Filter by Confidence Score ---
# Keep only detections with a confidence score above the threshold
detections = detections[detections.confidence >= CONFIDENCE_THRESHOLD]
print(f"Detections after Confidence Filter ({CONFIDENCE_THRESHOLD}): {len(detections)}")

print("detections.confidence:", detections.confidence)

Detections after Confidence Filter (0.75): 1
detections.confidence: [     0.8965]


In [15]:
# --- 4b. Filter by Bounding Box Size (Normalized Area) ---
if len(detections) > 0:
    # --- Re-calculate area for the remaining boxes for filtering ---
    xyxy = detections.xyxy
    box_widths = xyxy[:, 2] - xyxy[:, 0]
    box_heights = xyxy[:, 3] - xyxy[:, 1]
    
    # Calculate the absolute area of each bounding box
    box_areas = box_widths * box_heights # <--- We will use this array later
    
    # Calculate the normalized area ratio (Area / Total Image Area)
    normalized_areas = box_areas / TOTAL_IMAGE_AREA
    
    # Create a boolean filter mask
    area_filter = normalized_areas >= MIN_NORMALIZED_AREA_RATIO
    
    # Apply the area filter
    detections = detections[area_filter]

print(f"Detections after Area Filter ({MIN_NORMALIZED_AREA_RATIO*100}%): {len(detections)}")

Detections after Area Filter (10.0%): 1


In [56]:
import cv2
import numpy as np
import supervision as sv # Keep import for Detections object

if len(detections) > 0:

    # ============================
    # FIND MAIN (LARGEST) CAR
    # ============================
    xyxy = detections.xyxy

    box_widths  = xyxy[:, 2] - xyxy[:, 0]
    box_heights = xyxy[:, 3] - xyxy[:, 1]
    box_areas   = box_widths * box_heights

    largest_index = np.argmax(box_areas)
    # Using slice indexing [start:end] to preserve the 2D array structure required by supervision
    main_car_detection = detections[largest_index:largest_index+1] 

    print("\n✅ VALID CAR FOUND IN FOREGROUND.")

    annotated_image = image.copy() # Use image_original if available

    # ============================
    # DRAW BOUNDING BOX + LABEL
    # ============================
    for box, conf in zip(main_car_detection.xyxy, main_car_detection.confidence):

        x1, y1, x2, y2 = map(int, box)

        # Bounding box (RED)
        cv2.rectangle(annotated_image, (x1, y1), (x2, y2), (255,0,0), 2)

        # Label lines with the requested format
        line1 = f"Detected = car"
        line2 = f"confidence = {conf*100:.0f}%"
        
        # --- Multi-Line Text Placement ---
        
        TEXT_FONT = cv2.FONT_HERSHEY_SIMPLEX
        TEXT_SCALE = 0.7
        TEXT_THICKNESS = 2
        TEXT_PAD = 5
        LINE_SPACING = 5 # Vertical space between lines

        # 1. Measure the size of each line
        (w1, h1), _ = cv2.getTextSize(line1, TEXT_FONT, TEXT_SCALE, TEXT_THICKNESS)
        (w2, h2), _ = cv2.getTextSize(line2, TEXT_FONT, TEXT_SCALE, TEXT_THICKNESS)
        
        # 2. Determine the size of the final background rectangle
        max_w = max(w1, w2)
        total_h = h1 + h2 + LINE_SPACING # Total height for both lines
        
        # We will place the text block at the top-left corner of the bounding box
        text_x = x1
        text_y = y1 - total_h - TEXT_PAD # Positioned just above the box

        # Fallback if label goes off the top edge of the image
        if text_y < 0:
            text_y = y1 + h1 + TEXT_PAD # Place inside the box, below the top edge

        # 3. Background Rectangle (BLACK)
        cv2.rectangle(annotated_image, 
                      (text_x, text_y - h1 - TEXT_PAD), # Top-left corner
                      (text_x + max_w + TEXT_PAD*2, text_y + h2 + TEXT_PAD*2), # Bottom-right corner
                      (0,0,0), -1)

        # 4. Plot Line 1 (Detected = car)
        cv2.putText(annotated_image, line1, 
                    (text_x + TEXT_PAD, text_y - TEXT_PAD), # x-pos, y-pos
                    TEXT_FONT, TEXT_SCALE,
                    (255,50,255), TEXT_THICKNESS, cv2.LINE_AA) # White text

        # 5. Plot Line 2 (confidence = 90%)
        cv2.putText(annotated_image, line2, 
                    (text_x + TEXT_PAD, text_y + h2 + LINE_SPACING), # x-pos, y-pos
                    TEXT_FONT, TEXT_SCALE,
                    (255,50,255), TEXT_THICKNESS, cv2.LINE_AA) # White text


    # Save
    output_name = "output_valid_car_filtered_final.jpg"
    cv2.imwrite(output_name, annotated_image)
    print(f"📁 Output saved as {output_name}")

else:
    print("\n❌ NO VALID CAR FOUND.")


✅ VALID CAR FOUND IN FOREGROUND.
📁 Output saved as output_valid_car_filtered_final.jpg
